# 00 · Join UBIGEO y carga de fuentes de datos

Notebook de la Etapa 0 del pipeline: antes de tocar una sola imagen, se cargan y verifican todas las fuentes de datos y se construye la tabla maestra unida por UBIGEO.

## Setup

In [1]:
import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 50)


# Polígonos y ubicación geográfica

## Límite distrital INEI 2025 (shapefile)

Fuente: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

Descarga manual (no tiene link directo de descarga)

In [5]:
# Ruta local al shapefile ya descargado
ruta_limite_distrital = "../data/raw/Limite Distrital INEI 2025 CPV/Limite Distrital INEI 2025 CPV.shp"

limite_distrital = gpd.read_file(ruta_limite_distrital)
print(limite_distrital.shape)
limite_distrital.head()


(1891, 10)


,UBIGEO,CCDD,CCPP,CCDI,DEPARTAMEN,PROVINCIA,DISTRITO,OBJECTID,ESRI_OID,geometry
0,010101,01,01,01,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,5.0,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,01,01,02,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,6.0,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,01,01,03,AMAZONAS,CHACHAPOYAS,BALSAS,3.0,7.0,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,01,01,04,AMAZONAS,CHACHAPOYAS,CHETO,4.0,8.0,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,01,01,05,AMAZONAS,CHACHAPOYAS,CHILIQUIN,5.0,9.0,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


## Tabla de UBIGEO (crosswalk departamento-provincia-distrito)

Fuente (CSV público, se puede leer directo desde la URL):
https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv

In [ ]:
url_ubigeo = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

ubigeo = pd.read_csv(url_ubigeo, dtype={"inei": str, "reniec": str})

print(ubigeo.shape)
ubigeo.head()

(1893, 20)


,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema
0,010101,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chachapoyas,153.78,201.43711796072299,2338.0,-6.229444,-77.872778,0.279657,0.642361,9.034625,1.439875
1,010102,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Asunción,25.71,13.9634383508363,2823.0,-6.032500,-77.710833,0.558549,0.423032,36.519949,15.680750
2,010103,010103,AMAZONAS,CHACHAPOYAS,BALSAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Balsas,357.09,4.0465988966367004,859.0,-6.835833,-78.019722,0.646749,0.315308,45.732962,15.427120
3,010104,010104,AMAZONAS,CHACHAPOYAS,CHETO,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Cheto,56.97,13.568544848165701,2143.0,-6.255556,-77.700833,0.530846,0.345746,39.169782,23.678410
4,010105,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chiliquín,143.43,6.8326012689116604,2677.0,-6.078333,-77.737500,0.706854,0.275038,53.045662,36.395370


Como tenemos diferente cantidad de filas, verificaremos cuáles sonla que faltan en la 1ra

In [16]:
fila_nan = ubigeo[ubigeo["inei"].isna()]
fila_nan

,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema


In [15]:
# Quitar la fila con inei vacío (San Antonio, Moquegua - dato incompleto)
ubigeo = ubigeo[ubigeo["inei"].notna()].copy()
print("Distritos después de quitar el nan:", ubigeo.shape[0])

Distritos después de quitar el nan: 1892


In [17]:
# Los que están en el shapefile pero no en la tabla UBIGEO
print("--- En shapefile, no en tabla ---")
print(limite_distrital[limite_distrital["UBIGEO"].isin(["180107", "130112"])][["UBIGEO", "DEPARTAMEN", "PROVINCIA", "DISTRITO"]])

print()

# Los que están en la tabla UBIGEO pero no en el shapefile
print("--- En tabla, no en shapefile ---")
print(ubigeo[ubigeo["inei"].isin(["160109", "150144", "160114"])][["inei", "departamento", "provincia", "distrito"]])

--- En shapefile, no en tabla ---
      UBIGEO   DEPARTAMEN       PROVINCIA       DISTRITO
1182  130112  LA LIBERTAD        TRUJILLO  ALTO TRUJILLO
1534  180107     MOQUEGUA  MARISCAL NIETO    SAN ANTONIO

--- En tabla, no en shapefile ---
        inei departamento provincia                 distrito
1335  150144         LIMA      LIMA  SANTA MARIA DE HUACHIPA
1472  160109       LORETO    MAYNAS                 PUTUMAYO
1476  160114       LORETO    MAYNAS  TENIENTE MANUEL CLAVERO


In [18]:
ubigeos_finales = set(limite_distrital["UBIGEO"].astype(str)) & set(ubigeo["inei"].astype(str))
print("Distritos finales para el análisis:", len(ubigeos_finales))

limite_distrital = limite_distrital[limite_distrital["UBIGEO"].isin(ubigeos_finales)].copy()
ubigeo = ubigeo[ubigeo["inei"].isin(ubigeos_finales)].copy()

print("Shapefile final:", limite_distrital.shape[0])
print("Tabla ubigeo final:", ubigeo.shape[0])

Distritos finales para el análisis: 1889
Shapefile final: 1889
Tabla ubigeo final: 1889


# Imágenes y variables satelitales (Google Earth Engine)

## Inicializar Earth Engine

Requiere cuenta de Google Earth Engine aprobada: https://earthengine.google.com

In [20]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")



Successfully saved authorization token.
Google Earth Engine listo


## VIIRS Nighttime Lights

Dataset GEE: `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`

In [21]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").select("avg_rad")
viirs_mean = viirs.mean()
print(viirs.first().getInfo())


{'type': 'Image', 'bands': [{'id': 'avg_rad', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [86400, 33600], 'crs': 'EPSG:4326', 'crs_transform': [0.0041666667, 0, -180.00208525335, 0, -0.0041666667, 75.00208393335001]}], 'version': 1494360354199000.0, 'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG/20140101', 'properties': {'system:time_start': 1388534400000, 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'system:time_end': 1391212800000, 'system:asset_size': 13625191512, 'system:index': '20140101'}}


## ESA WorldCover

Dataset GEE: `ESA/WorldCover/v200` — cobertura de suelo (% construido, % cultivo, % árboles).

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
print(worldcover.bandNames().getInfo())


## SRTM — elevación y pendiente del terreno

Dataset GEE: `USGS/SRTMGL1_003`

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(srtm)
print(srtm.bandNames().getInfo())


## Sentinel-2 L2A

Dataset GEE: `COPERNICUS/S2_SR_HARMONIZED` — serie temporal, 12 bandas, 10m.

In [ ]:
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
)
print("Número de imágenes:", sentinel2.size().getInfo())


## Planet NICFI

Registro gratuito para el trópico: https://www.planet.com/nicfi/

Requiere API key propia de Planet, no se accede vía Earth Engine público.

In [ ]:
planet_api_key = ""  # pegar aquí la API key de Planet NICFI

headers = {"Authorization": f"api-key {planet_api_key}"}
url_nicfi = "https://api.planet.com/basemaps/v1/mosaics"

# response = requests.get(url_nicfi, headers=headers)
# response.json()


## Google Open Buildings V3

Fuente: https://sites.research.google/gr/open-buildings/

Se puede acceder también como tabla en Earth Engine: `GOOGLE/Research/open-buildings/v3/polygons`

In [ ]:
open_buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
print(open_buildings.limit(5).getInfo())


# Datos administrativos y municipales

## RENAMU (Registro Nacional de Municipalidades)

**Fuente:** https://proyectos.inei.gob.pe/microdatos/ (buscar "RENAMU" — desde 2021 la encuesta se estandariza en un único módulo llamado "Registro Nacional de Municipalidades - RENAMU", sin división por módulos como en años anteriores)

RENAMU es la encuesta anual que aplica el INEI a todas las municipalidades del país. Acá se usa como fuente de variables de control para el modelo causal (Etapa 6, DML + Causal Forest): el riesgo que cubre es que el modelo confunda "el gasto no rinde porque el territorio no responde" con "el gasto no rinde porque la municipalidad gestiona mal". Sin este control, τ podría estar capturando capacidad de gestión municipal en vez de contexto territorial — que es justo lo que el proyecto busca aislar.

### Por qué el panel arranca en 2021 y no en 2020

- El módulo con la pregunta sobre anemia (P68_7) recién aparece en la encuesta 2021. RENAMU 2020 no la tiene, así que no sirve como punto de partida.
- Desde 2021 el formato está estandarizado en un único módulo: mismos nombres de columna año a año (confirmado revisando el portal directamente, carpeta por carpeta).
- Desde 2021 el propio CSV trae la columna `Ubigeo` ya limpia en la cabecera. En 2020 solo existe `idmunici`, y hay que reconstruir el ubigeo a mano con `.zfill(6)` — trabajo extra que no vale la pena si igual ese año no trae la variable que más importa.

Por eso el panel se arma como loop 2021 → último año disponible, con el mismo código para cada año, en vez de tratar 2020 como caso especial.

### Variables extraídas (panel por Ubigeo + Año)

| Variable | Código original | Nombre final | Redacción exacta del diccionario (RENAMU 2021) | Corrección de año |
|---|---|---|---|---|
| Personal municipal | P19D_T | `personal_total` | *"Total personal / 31 de diciembre 2020"* — dentro del bloque "Personal de la municipalidad, al 31 de diciembre 2020". Es un conteo, no Sí/No. | **Sí, retrospectiva.** El campo describe el personal al cierre del año anterior a la encuesta, no del año de la encuesta. Etiquetar como año = X−1 |
| Programa de prevención de anemia con MINSA | P68_7 | `programa_anemia` | *"En el año 2020, ¿La municipalidad implementó programas de control y prevención de la salud en coordinación con el MINSA en: Prevención y reducción de la anemia"* — ítem dentro de un checklist de 11 opciones (P68_1 a P68_11), no un Sí/No binario simple. Verificar valores únicos reales en el CSV antes de tratarla como booleana. | Sí, retrospectiva: encuesta año X pregunta por lo ejecutado en X−1. Etiquetar como año = X−1 |
| Centro de salud administrado por la municipalidad | P66_2 | `centro_salud_municipal` | *"¿En el Distrito funcionan establecimientos de salud administrados por la municipalidad: Centro de salud?"* — 1: Sí / 2: No | No. Pregunta en tiempo presente, sin referencia retrospectiva → el valor corresponde al mismo año de la encuesta |

### Corrección de año: las tres variables no llevan el mismo tratamiento

A diferencia de lo que se asumió inicialmente, **dos de las tres variables son retrospectivas, no solo una**:

- `personal_total` (P19D_T) y `programa_anemia` (P68_7) describen la situación del año **anterior** a la encuesta → se etiquetan con año = Año_encuesta − 1.
- `centro_salud_municipal` (P66_2) describe la situación **al momento de la encuesta** → se etiqueta con año = Año_encuesta, sin ajuste.

Si esta corrección no se aplica correctamente a cada variable, el merge con anemia (SIEN) y gasto (SIAF) queda desfasado, y el modelo terminaría comparando información municipal de un año con el gasto/anemia de otro año distinto.

### Pendiente antes de dar por cerrada la tabla

- Confirmar con `df["programa_anemia"].value_counts()` qué valores únicos trae realmente esa columna en el CSV cargado — el diccionario sugiere que es un ítem de checklist (0/Pase, 9/Sí), no un Sí/No de dos valores limpio como se pensaba.

In [34]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)

    if not carpeta.exists():
        print(f"⚠️ {año}: no existe la carpeta {carpeta}, saltando")
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]

    if len(csvs) != 1:
        print(f"⚠️ {año}: encontré {len(csvs)} CSV en la carpeta, revisar manualmente")
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    faltantes = [c for c in columnas_necesarias if c not in df.columns]
    if faltantes:
        print(f"⚠️ {año}: faltan columnas {faltantes} — revisar nombre exacto en el diccionario {año}")
        print(f"   Columnas disponibles (primeras 15): {list(df.columns)[:15]}")
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    # Diagnóstico rápido: confirmar qué valores trae programa_anemia antes de asumir Sí/No
    print(f"{año} — valores únicos en programa_anemia: {df['programa_anemia'].unique()}")

    paneles.append(df)
    print(f"✅ {año}: {df.shape[0]} municipalidades cargadas")

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Corrección de año: personal_total y programa_anemia son retrospectivas ---
# (describen el año anterior a la encuesta); centro_salud_municipal no lo es.
renamu_panel["Año_personal_total"] = renamu_panel["Año_encuesta"] - 1
renamu_panel["Año_programa_anemia"] = renamu_panel["Año_encuesta"] - 1
# centro_salud_municipal usa Año_encuesta directamente, sin columna adicional

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2021 — valores únicos en programa_anemia: [0 7]
✅ 2021: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2022 — valores únicos en programa_anemia: [0 7]
✅ 2022: 1874 municipalidades cargadas


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


2023 — valores únicos en programa_anemia: [ 7.  0. nan]
✅ 2023: 1891 municipalidades cargadas
2024 — valores únicos en programa_anemia: [0 7]
✅ 2024: 1891 municipalidades cargadas
(7530, 7)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2925557397.py:31: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta,Año_personal_total,Año_programa_anemia
0,010101,208.0,0.0,2.0,2021,2020,2020
1,010102,1.0,0.0,2.0,2021,2020,2020
2,010103,7.0,0.0,2.0,2021,2020,2020
3,010104,5.0,7.0,2.0,2021,2020,2020
4,010105,2.0,0.0,2.0,2021,2020,2020


In [36]:
from pathlib import Path
import pandas as pd

base_dir = Path("../data/raw/RENAMU")
años = range(2021, 2025)  # ajustar según lo que confirmes disponible en el portal

columnas_necesarias = {
    "Ubigeo": "ubigeo",
    "P19D_T": "personal_total",
    "P68_7": "programa_anemia",
    "P66_2": "centro_salud_municipal",
}

nulos_renamu = ["#Â¡NULO!", "#¡NULO!"]

paneles = []

for año in años:
    carpeta = base_dir / str(año)
    if not carpeta.exists():
        continue

    csvs = [f for f in carpeta.iterdir() if f.suffix.lower() == ".csv"]
    if len(csvs) != 1:
        continue

    df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)

    if any(c not in df.columns for c in columnas_necesarias):
        continue

    df = df[list(columnas_necesarias.keys())].rename(columns=columnas_necesarias)
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["Año_encuesta"] = año

    paneles.append(df)

renamu_panel = pd.concat(paneles, ignore_index=True)

# --- Tres tablas filtradas, cada una con SOLO su variable y SU año real ---
# (listas para mergear después, cuando tengas tabla_maestra armada)

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.shape)
renamu_panel.head()

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_4_4_O, 1: P37A_5_O, 2: P73_4_O, 3: P95_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_1_4_O, 1: P23_4_4_O, 2: P23_9_4_O, 3: P23_11_4_O, 4: P31_5_O, 5: P37A_5_O, 6: P73_4_O, 7: P79A_5_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)
C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_14_1_O, 1: P23_14_2, 2: P28_5_O, 3: P37A_5_O, 4: P48_12_O, 5: P55_7_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


(7530, 5)


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_8288\2681341985.py:27: DtypeWarning: Columns (0: P23_13_2, 1: P25_4_O, 2: P28_5_O, 3: P48_12_O) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csvs[0], sep=";", encoding="latin-1", na_values=nulos_renamu)


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208.0,0.0,2.0,2021
1,010102,1.0,0.0,2.0,2021
2,010103,7.0,0.0,2.0,2021
3,010104,5.0,7.0,2.0,2021
4,010105,2.0,0.0,2.0,2021


### Notas de codificación y tipos de dato

**`centro_salud_municipal` (P66_2):** es un indicador binario (Sí/No), no una cantidad. Corresponde específicamente al tipo de establecimiento "Centro de salud" administrado por la municipalidad — no incluye hospitales, postas, consultorios ni otros tipos (esos son preguntas separadas en el diccionario: P66_1, P66_3, P66_4, etc.). El campo de conteo real (`P66_2_1`, "Número de establecimientos") no forma parte de este panel.

**`programa_anemia` (P68_7):** el diccionario de RENAMU codifica esta pregunta como parte de un checklist de 11 programas de salud (P68_1 a P68_11), donde cada ítem usa su propia posición como código de "Sí" en vez de un 1/2 estándar. Para P68_7 específicamente: `0 = Pase` (no marcó esta opción) y `7 = Sí` (sí implementó el programa de anemia). Antes de usar esta variable en el modelo causal, se recodifica a booleano estándar (1 = Sí, 0 = No) para que no se interprete como una magnitud numérica.

**Tipos de dato:** las tres variables llegan como `float64` por los valores nulos (`NaN`) presentes en el CSV crudo. Se mantienen como `float64` hasta el merge final — convertir a `int` antes de imputar/tratar los nulos generaría un error, porque `NaN` no es representable como entero en pandas.

In [37]:
# --- Recodificación a booleano estándar (1 = Sí, 0 = No) ---

# programa_anemia: el diccionario de RENAMU codifica "Sí" como 7 (la posición
# del ítem "anemia" dentro del checklist P68_1 a P68_11), no como 1.
# Se recodifica para que el modelo no lo interprete como una magnitud.
renamu_panel["programa_anemia"] = (renamu_panel["programa_anemia"] == 7).astype("Int64")

# centro_salud_municipal: viene como 1=Sí, 2=No (estándar RENAMU).
# Se recodifica a 1=Sí, 0=No para mantener consistencia con programa_anemia.
renamu_panel["centro_salud_municipal"] = (renamu_panel["centro_salud_municipal"] == 1).astype("Int64")

# personal_total: es un conteo real (no booleano). Se pasa a entero nullable
# porque tiene NaN, y un int64 normal de numpy no admite nulos.
renamu_panel["personal_total"] = renamu_panel["personal_total"].astype("Int64")

# --- Reconstruir las tres tablas filtradas con los valores ya recodificados ---

centro_salud = renamu_panel[["ubigeo", "Año_encuesta", "centro_salud_municipal"]].rename(
    columns={"Año_encuesta": "Año"}
)

personal = renamu_panel[["ubigeo", "Año_encuesta", "personal_total"]].copy()
personal["Año"] = personal["Año_encuesta"] - 1
personal = personal[["ubigeo", "Año", "personal_total"]]

anemia_muni = renamu_panel[["ubigeo", "Año_encuesta", "programa_anemia"]].copy()
anemia_muni["Año"] = anemia_muni["Año_encuesta"] - 1
anemia_muni = anemia_muni[["ubigeo", "Año", "programa_anemia"]]

print(renamu_panel.dtypes)
renamu_panel.head()

ubigeo                      str
personal_total            Int64
programa_anemia           Int64
centro_salud_municipal    Int64
Año_encuesta              int64
dtype: object


,ubigeo,personal_total,programa_anemia,centro_salud_municipal,Año_encuesta
0,010101,208,0,0,2021
1,010102,1,0,0,2021
2,010103,7,0,0,2021
3,010104,5,1,0,2021
4,010105,2,0,0,2021


## ENAHO (Encuesta Nacional de Hogares)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENAHO", módulos 1, 34, 200, 300, 400, 500)

In [ ]:
ruta_enaho_hogar = ""    # ej: "data/raw/ENAHO_1_hogar.csv"
ruta_enaho_sumaria = ""  # ej: "data/raw/ENAHO_34_sumaria.csv"

enaho_hogar = pd.read_csv(ruta_enaho_hogar)
enaho_sumaria = pd.read_csv(ruta_enaho_sumaria)

print(enaho_hogar.shape, enaho_sumaria.shape)
enaho_hogar.head()


## ENDES (Encuesta Demográfica y de Salud Familiar)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENDES", módulos RECH0 y RECH23)

In [ ]:
ruta_endes_rech0 = ""   # ej: "data/raw/ENDES_1629_RECH0.csv"
ruta_endes_rech23 = ""  # ej: "data/raw/ENDES_1630_RECH23.csv"

endes_rech0 = pd.read_csv(ruta_endes_rech0)
endes_rech23 = pd.read_csv(ruta_endes_rech23)

print(endes_rech0.shape, endes_rech23.shape)
endes_rech0.head()


# Anemia (registro administrativo)

## SIEN / REUNIS (MINSA)

Fuente: https://www.minsa.gob.pe/reunis/

No tiene descarga directa en CSV — requiere revisar el tablero o solicitar la data en otro formato.

In [24]:
ruta_sien = "../data/raw/Minsa_reunis_anemia/Trama_Base_Anemia.xlsx"

sien = pd.read_excel(ruta_sien, dtype={"Ubigeo": str})
print(sien.shape)
sien.head()


(355876, 15)


,Año,Edad,Diresa,Departamento,Provincia,Distrito,Renipres,Ubigeo,Sexo,Evaluados,Anemia,Anemia Leve,Anemia Moderada,Anemia Severa,Normal
0,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,M,12,0,0,0,0,12
1,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,F,14,0,0,0,0,14
2,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,M,3,0,0,0,0,3
3,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,F,4,0,0,0,0,4
4,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5055.0,010202,M,6,0,0,0,0,6


In [28]:
# Filtrar solo menores de 3 años (Edad == 1), según confirmamos contra el tablero
sien_menores_3 = sien[sien["Edad"] == 1].copy()

anemia_distrital = (
    sien_menores_3
    .groupby(["ubigeo", "Año"], as_index=False)
    .agg(
        departamento=("Departamento", "first"),
        provincia=("Provincia", "first"),
        distrito=("Distrito", "first"),
        ninos_evaluados=("Evaluados", "sum"),
        ninos_con_anemia=("Anemia", "sum"),
        ninos_sin_anemia=("Normal", "sum"),
    )
)

anemia_distrital["prevalencia_anemia"] = (
    anemia_distrital["ninos_con_anemia"] / anemia_distrital["ninos_evaluados"]
)

print(anemia_distrital.shape)
anemia_distrital.head()

(13029, 9)


,ubigeo,Año,departamento,provincia,distrito,ninos_evaluados,ninos_con_anemia,ninos_sin_anemia,prevalencia_anemia
0,010101,2020,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,245,85,160,0.346939
1,010101,2021,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,234,71,163,0.303419
2,010101,2022,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,201,76,125,0.378109
3,010101,2023,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,837,160,677,0.191159
4,010101,2024,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,969,157,812,0.162023


In [ ]:
print(sorted(sien["Año"].unique()))
# tebemos datos desde el 2020

[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## Datos Abiertos — Anemia (MINSA / INS)

Fuente: https://datosabiertos.gob.pe/dataset/anemia

(No los tomamos porque simplemente reunen otras datas que ya se repitieron o son muy específicas, por ejemplo para ayacucho o por años)

In [30]:
# ruta_anemia_datosabiertos = ""  # ej: "data/raw/anemia_datosabiertos.csv"


## Presupuesto público

### SIAF — Gasto Devengado Mensual (Portal de Datos Abiertos MEF)

Fuente: https://datosabiertos.mef.gob.pe/dataset/presupuesto-y-ejecucion-de-gasto-devengado-mensual

Descarga directa en CSV (un archivo por año) — ya no requiere scraping del Consulta Amigable, el MEF lo publica en formato abierto.

Se usa el **Devengado** (no Compromiso ni Girado) porque es la fase contable que refleja el gasto efectivamente ejecutado: el bien o servicio ya fue recibido y la obligación de pago ya está reconocida — es el estándar usado para medir "cuánto se gastó realmente".

Se descargan los años **2020 a 2026**, para que coincidan con el rango de años disponible en la base de anemia (SIEN)

In [ ]:
ruta_siaf = ""  # ej: "data/raw/siaf_gasto_distrital.csv"

siaf = pd.read_csv(ruta_siaf)
print(siaf.shape)
siaf.head()


# Join final por UBIGEO

Una vez cargadas todas las fuentes, se unen por la llave `ubigeo` para construir la tabla maestra distrital.

In [ ]:
# maestro_distritos = (
#     ubigeo
#     .merge(renamu, on="ubigeo", how="left")
#     .merge(enaho_sumaria, on="ubigeo", how="left")
#     .merge(endes_rech0, on="ubigeo", how="left")
#     .merge(sien, on="ubigeo", how="left")
#     .merge(anemia_da, on="ubigeo", how="left")
#     .merge(siaf, on="ubigeo", how="left")
# )
#
# print("Distritos en maestro:", maestro_distritos.shape[0])
# print("Duplicados UBIGEO:", maestro_distritos["ubigeo"].duplicated().sum())
# maestro_distritos.head()


## Guardar tabla maestra

In [ ]:
ruta_salida = ""  # ej: "data/clean/maestro_distritos_2025.csv"

# maestro_distritos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
# print("Archivo guardado en:", ruta_salida)
